# Practical 9 — Telecom Customer Data Cleaning

Create intentionally messy telecom data, then remove duplicates, fix text and data types, handle missing values/outliers, encode and scale features.

**Run the cells from top to bottom.** The dataset is generated inside this notebook before the practical begins.

## 1. Install / import requirements

If a package is missing, run `!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels openpyxl requests xgboost` in a notebook cell. Most environments already have the common packages.

In [1]:
from pathlib import Path
Path('data').mkdir(exist_ok=True)


## 2. Create the required dataset

This is the relevant part of `make_data.py`, placed here so the notebook is self-contained.

In [2]:
# Create a deliberately untidy telecom churn dataset for cleaning practice.
import numpy as np, pandas as pd
rng = np.random.default_rng(42)
n=1000
customers=pd.DataFrame({"customerID":range(n),
 "gender":rng.choice(["Male","Female","male","FEMALE"],n),
 "tenure":rng.integers(0,72,n), "MonthlyCharges":rng.uniform(20,120,n).round(2),
 "Contract":rng.choice(["Month-to-month","One year","Two year"],n,p=[.55,.25,.2])})
customers["TotalCharges"]=(customers.tenure*customers.MonthlyCharges).round(2).astype(object)
customers["Churn"]=np.where((customers.Contract=="Month-to-month")&(rng.random(n)<.5),"Yes","No")
customers.loc[rng.choice(n,30,replace=False),"TotalCharges"]=" "
customers.loc[rng.choice(n,20,replace=False),"MonthlyCharges"]=np.nan
customers.loc[rng.choice(n,8,replace=False),"MonthlyCharges"]=999
pd.concat([customers,customers.iloc[:15]]).to_csv("data/Telecom_Customer_Churn.csv",index=False)
print("Created data/Telecom_Customer_Churn.csv")


Created data/Telecom_Customer_Churn.csv


## 3. Practical implementation

The following cells perform the analysis and show the output.

In [3]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/Telecom_Customer_Churn.csv")                 # 1. import
print(df.shape); print(df.info()); print(df.isna().sum())            # 2. explore

df = df.drop_duplicates()                                            # 4. duplicates
df["gender"] = df.gender.str.title()                                 # 5. fix inconsistent text (male/MALE -> Male)
df["TotalCharges"] = pd.to_numeric(df.TotalCharges, errors="coerce") # 6. correct dtype (" " -> NaN)

df["MonthlyCharges"] = df.MonthlyCharges.fillna(df.MonthlyCharges.median())   # 3. missing values
df["TotalCharges"] = df.TotalCharges.fillna(df.TotalCharges.median())

q1, q3 = df.MonthlyCharges.quantile([.25, .75])                      # 7. outliers (IQR)
df = df[df.MonthlyCharges.between(q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))]

df["AvgChargePerMonth"] = df.TotalCharges / (df.tenure + 1)          # 8. feature engineering
df["IsLongTerm"] = (df.Contract != "Month-to-month").astype(int)
df["Churn"] = df.Churn.map({"Yes": 1, "No": 0})

df = pd.get_dummies(df.drop(columns="customerID"), drop_first=True)  # encode categoricals
num = ["tenure", "MonthlyCharges", "TotalCharges", "AvgChargePerMonth"]
df[num] = StandardScaler().fit_transform(df[num])                    # 9. scale

X, y = df.drop(columns="Churn"), df.Churn                            # 10. split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
print("Train/test:", X_train.shape, X_test.shape)

df.to_csv("data/Telecom_Cleaned.csv", index=False)                   # 11. export
print("Saved data/Telecom_Cleaned.csv")

(1015, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customerID      1015 non-null   int64  
 1   gender          1015 non-null   object 
 2   tenure          1015 non-null   int64  
 3   MonthlyCharges  995 non-null    float64
 4   Contract        1015 non-null   object 
 5   TotalCharges    1015 non-null   object 
 6   Churn           1015 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 55.6+ KB
None
customerID         0
gender             0
tenure             0
MonthlyCharges    20
Contract           0
TotalCharges       0
Churn              0
dtype: int64
Train/test: (793, 8) (199, 8)
Saved data/Telecom_Cleaned.csv
